Purpose: perform chronological train/val/test split on engineered data.

Load engineered dataset and define split helper.

In [1]:
import pandas as pd
from pathlib import Path

fe_path = Path('artifacts/feature_engineered.csv')
if not fe_path.exists():
    raise FileNotFoundError('Missing artifacts/feature_engineered.csv - run feature engineering first')

df = pd.read_csv(fe_path, parse_dates=['datetime']).sort_values('datetime').set_index('datetime')

candidate_targets = ['CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)']
targets = [c for c in candidate_targets if c in df.columns]
print('Targets for splitting:', targets)

feature_cols = [c for c in df.columns if c not in targets]

def chronological_split(df, target_cols, test_year=2005, val_ratio=0.15):
    train_val = df[df.index.year < test_year]
    test = df[df.index.year == test_year]
    n_val = int(len(train_val) * val_ratio)
    train = train_val.iloc[:-n_val] if n_val > 0 else train_val
    val = train_val.iloc[-n_val:] if n_val > 0 else pd.DataFrame()
    return {
        'X_train': train[feature_cols],
        'y_train': train[target_cols],
        'X_val': val[feature_cols],
        'y_val': val[target_cols] if len(val) else pd.DataFrame(columns=target_cols),
        'X_test': test[feature_cols],
        'y_test': test[target_cols],
    }

splits = chronological_split(df, targets)
for k in ['X_train','X_val','X_test']:
    print(k, splits[k].shape)


Targets for splitting: ['CO(GT)', 'NMHC(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)']
X_train (6024, 102)
X_val (1062, 102)
X_test (2247, 102)


Save split datasets to artifacts.

In [2]:
out_dir = Path('artifacts')
(out_dir / '').mkdir(exist_ok=True)

splits['X_train'].to_csv(out_dir / 'train_features.csv')
splits['y_train'].to_csv(out_dir / 'train_targets.csv')
splits['X_val'].to_csv(out_dir / 'val_features.csv')
splits['y_val'].to_csv(out_dir / 'val_targets.csv')
splits['X_test'].to_csv(out_dir / 'test_features.csv')
splits['y_test'].to_csv(out_dir / 'test_targets.csv')

print('Saved split datasets to artifacts/')


Saved split datasets to artifacts/
